<a href="https://colab.research.google.com/github/aditya-r-m/experimental/blob/rl/reinforcement-learning/square.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from typing import Tuple, List, Dict, Set
from itertools import product

class IDict(dict):
    __missing__ = lambda self, key : key

def compose(state: Tuple[int, ...], move: Dict[int, int]):
    next_state = [None] * len(state)
    for i in range(len(state)):
        next_state[move[i]] = state[i]
    return tuple(next_state)

base_state: Tuple[int, ...] = tuple(e for e in range(4) for _ in range(3))

moves: List[Dict[int, int]] = [
    IDict({}),
    IDict({0: 8, 8: 0, 9: 11, 11: 9}),
    IDict({2: 6, 6: 2, 3: 5, 5: 3}),
    IDict({3: 11, 11: 3, 0: 2, 2: 0}),
    IDict({5: 9, 9: 5, 6: 8, 8: 6}),
]

def make_move(
    state: Tuple[int],
    move: Dict[int, int],
):
    next_state = compose(state, move)
    reward = sum(
        map(lambda x: int(x[0] == x[1]),
            zip(base_state, next_state))
    )
    return (next_state, reward)

states: Set[Tuple] = set()

for chain in product(*[moves]*5):
  cur_state = base_state
  for move in chain: cur_state = compose(cur_state, move)
  states.add(cur_state)

print(base_state)
print(moves)
print(len(states))


(0, 0, 0, 1, 1, 1, 2, 2, 2, 3, 3, 3)
[{}, {0: 8, 8: 0, 9: 11, 11: 9}, {2: 6, 6: 2, 3: 5, 5: 3}, {3: 11, 11: 3, 0: 2, 2: 0}, {5: 9, 9: 5, 6: 8, 8: 6}]
24


In [ ]:
from collections import defaultdict
from typing import DefaultDict, Tuple

from random import randrange

q: DefaultDict[Tuple[Tuple, Tuple, int], float] = defaultdict(lambda: 0.0)

learning_rate = 0.01
discount_factor = 0.9
training_episode_count = 10000

def update_q(
    state: Tuple,
    m: int,
    next_state: Tuple,
    reward: float,
):
    q[(state, m)] = (1 - learning_rate) * q[
        (state, m)
    ] + learning_rate * (
        reward + max(q[(next_state, n)] for n in range(len(moves)))
    )

def train():
  for _ in range(training_episode_count):
    for state in states:
      for m in range(len(moves)):
          next_state, reward = make_move(state, moves[m])
          update_q(state, m, next_state, reward)

train()

def greedy_policy(state):
  return max((q[(state, m)], m) for m in range(len(moves)))[1]

for state in states:
  original_state = state
  for steps in range(5):
    if state == base_state: break
    state = compose(state, moves[greedy_policy(state)])
  print(steps, state, '<-', original_state)


3 (0, 0, 0, 1, 1, 1, 2, 2, 2, 3, 3, 3) <- (0, 0, 2, 3, 1, 3, 0, 2, 2, 1, 3, 1)
4 (0, 0, 0, 1, 1, 1, 2, 2, 2, 3, 3, 3) <- (2, 0, 2, 3, 1, 3, 0, 2, 0, 1, 3, 1)
1 (0, 0, 0, 1, 1, 1, 2, 2, 2, 3, 3, 3) <- (0, 0, 0, 1, 1, 3, 2, 2, 2, 1, 3, 3)
1 (0, 0, 0, 1, 1, 1, 2, 2, 2, 3, 3, 3) <- (0, 0, 2, 1, 1, 1, 0, 2, 2, 3, 3, 3)
2 (0, 0, 0, 1, 1, 1, 2, 2, 2, 3, 3, 3) <- (2, 0, 0, 3, 1, 1, 0, 2, 2, 3, 3, 1)
2 (0, 0, 0, 1, 1, 1, 2, 2, 2, 3, 3, 3) <- (0, 0, 2, 3, 1, 1, 2, 2, 0, 3, 3, 1)
2 (0, 0, 0, 1, 1, 1, 2, 2, 2, 3, 3, 3) <- (0, 0, 2, 1, 1, 3, 0, 2, 2, 3, 3, 1)
3 (0, 0, 0, 1, 1, 1, 2, 2, 2, 3, 3, 3) <- (2, 0, 0, 3, 1, 3, 2, 2, 0, 1, 3, 1)
0 (0, 0, 0, 1, 1, 1, 2, 2, 2, 3, 3, 3) <- (0, 0, 0, 1, 1, 1, 2, 2, 2, 3, 3, 3)
2 (0, 0, 0, 1, 1, 1, 2, 2, 2, 3, 3, 3) <- (0, 0, 0, 3, 1, 3, 2, 2, 2, 1, 3, 1)
3 (0, 0, 0, 1, 1, 1, 2, 2, 2, 3, 3, 3) <- (2, 0, 0, 3, 1, 1, 0, 2, 2, 1, 3, 3)
2 (0, 0, 0, 1, 1, 1, 2, 2, 2, 3, 3, 3) <- (2, 0, 0, 3, 1, 1, 2, 2, 0, 1, 3, 3)
2 (0, 0, 0, 1, 1, 1, 2, 2, 2, 3, 3, 3) <- (2, 0, 2, 